# DNT Organoid AI — High-Memory Cloud Quantification Bridge
## Reference Index Preparation & Persistent Storage Synchronization

**Study Accession:** BioProject `PRJNA802072` (SRA `SRP357550`)
**Target Runtime:** Google Colab High-RAM Instance (83.5 GB RAM, 12 vCPUs, ~235 GB Local NVMe/SSD)
**Storage Architecture:** Local NVMe/SSD for High-IOPS Active Processing + Google Drive for Persistent Verification Checkpoints
**Reference Build:** Ensembl GRCh38 Release 112 (Primary Assembly + Annotation GTF + Deterministic BED12)

> **GOVERNANCE MANDATE:**
> 1. **Zero Premature Module Imports:** Google Drive mount and runtime setup rely strictly on standard library until repository code is cloned.
> 2. **Local Scratch for Active Work:** Active STAR indexing and sorting execute on local NVMe (`/content`) to avoid Google Drive FUSE latency and rate limits.
> 3. **Verified Account Quota:** Google Drive storage capacity is verified via authorized API or explicit user confirmation against actual available headroom before transfers (never estimated from folder scans or hardcoded assumptions).
> 4. **Single-Archive Verified Persistence:** The completed STAR index is archived into a single verified tarball with a companion manifest, eliminating Drive FUSE rate limits.
> 5. **Non-Persistent Git Authentication:** Ephemeral in-memory askpass ensures tokens are never written to disk, CLI args, logs, or repository config.
> 6. **Scientific Review Gate:** Processing strictly pauses after reference-index generation and validation. No FASTQ sequencing libraries are downloaded until review is complete.

### Stage 1: Google Drive Mount & Persistent Storage Initialization
Mount Google Drive after user authorization. This stage relies strictly on standard library modules.

In [3]:
import os
import sys
from pathlib import Path

# Detect Colab runtime
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    print("Requesting Google Drive mount authorization...")
    drive.mount('/content/drive')
    PERSISTENCE_ROOT = Path('/content/drive/MyDrive/DNT_Organoid_AI_Persistence')
    print(f"[OK] Google Drive mounted. Persistent storage root: {PERSISTENCE_ROOT}")
else:
    PERSISTENCE_ROOT = Path('/tmp/DNT_Organoid_AI_Persistence')
    print(f"Running outside Colab. Fallback persistent storage root: {PERSISTENCE_ROOT}")

PERSISTENCE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Persistent directory initialized: {PERSISTENCE_ROOT}")


Requesting Google Drive mount authorization...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Google Drive mounted. Persistent storage root: /content/drive/MyDrive/DNT_Organoid_AI_Persistence
Persistent directory initialized: /content/drive/MyDrive/DNT_Organoid_AI_Persistence


### Stage 2: Fast Bioinformatics Toolchain Provisioning (Micromamba)
Install the pinned bioinformatics toolchain into Colab runtime:
- **STAR 2.7.11b** (Splice-aware reference aligner)
- **fastp 0.23.4** (Adapter and poly-G trimmer)
- **subread 2.0.6 / featureCounts** (Gene-level summarization)
- **samtools 1.19.2** (BAM sorting and indexing)
- **rseqc 5.0.2 / infer_experiment.py** (Strandedness verification)

In [5]:
%%bash
set -eo pipefail

# Fast micromamba provisioning in /content/micromamba
if [ ! -f "/content/micromamba/bin/micromamba" ]; then
  echo "Downloading standalone micromamba binary..."
  mkdir -p /content/micromamba
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/micromamba bin/micromamba
fi

if [ ! -d "/content/dnt_env" ]; then
  echo "Creating pinned bioinformatics environment (/content/dnt_env)..."
  /content/micromamba/bin/micromamba create -y -p /content/dnt_env \
    -c conda-forge -c bioconda \
    star=2.7.11b \
    fastp=0.23.4 \
    subread=2.0.6 \
    samtools=1.19.2 \
    rseqc=5.0.3 \
    python=3.12 \
    pyyaml \
    pandas \
    pytest
fi

# Test tools in environment
export PATH="/content/dnt_env/bin:$PATH"
echo "Toolchain Verification:"
STAR --version
fastp --version
featureCounts -v 2>&1 | head -n 2
samtools --version | head -n 1
infer_experiment.py --version 2>&1
echo "[OK] Bioinformatics toolchain successfully provisioned and verified."


Creating pinned bioinformatics environment (/content/dnt_env)...
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                           ⧖ Starting
Fetching and Parsing Packages' Shards                                                     ✔ Done (1.5 sec)
Using Cached Shard Index for bioconda/linux-64                                                      ✔ Done
Using Cached Shard Index for bioconda/noarch                                                        ✔ Done
Fetching and Parsing Packages' Shards                                                           ⧖ Starting
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.0 sec)
Using Cached Shard Index for conda-forge/linux-64                              

warning  libmamba Security Warning: This transaction includes executing package scripts (pre/post-link/unlink) if present. These scripts can contain arbitrary code. Please ensure you trust the package sources.
warning  libmamba [rseqc-5.0.3-py312hf67a6ed_3] The following files were already present in the environment:
    - lib/python3.12/site-packages/__pycache__/psyco_full.cpython-312.pyc
    - lib/python3.12/site-packages/psyco_full.py
fastp 0.23.4


### Stage 3: Secure Repository Setup on Local High-Speed SSD
Clone the approved development branch (`feature/fastq-sra-real-data-validation`) to `/content/DNT-Organoid-AI`.
Authentication uses an ephemeral, executable `GIT_ASKPASS` helper (`0700`) with shebang declaration, ensuring tokens are never written to Git configuration, clone URLs, notebook outputs, or saved files.

In [6]:
import os
import sys
import stat
import subprocess
from pathlib import Path

# Prepend conda environment to PATH
os.environ["PATH"] = f"/content/dnt_env/bin:{os.environ.get('PATH', '')}"

REPO_DIR = Path("/content/DNT-Organoid-AI")
BRANCH = "feature/fastq-sra-real-data-validation"
REPO_URL = "https://github.com/bioprintingLabsInc/DNT-Organoid-AI.git"

if not REPO_DIR.exists():
    print(f"Cloning DNT-Organoid-AI repository ({BRANCH}) to local SSD...")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass

    # Attempt public clone first
    res = subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], capture_output=True, text=True)

    if res.returncode != 0 and not token:
        import getpass
        print("Public clone was restricted. Please enter GitHub Access Token (hidden non-echoing input):")
        token = getpass.getpass("Token: ")

    if res.returncode != 0 and token:
        # Use ephemeral GIT_ASKPASS helper with valid shebang and 0700 permissions
        askpass_script = Path("/tmp/dnt_git_askpass.py")
        askpass_content = (
            f"#!{sys.executable}\n"
            "import sys, os\n"
            "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
            "if 'username' in prompt:\n"
            "    sys.stdout.write('x-access-token\\n')\n"
            "else:\n"
            "    sys.stdout.write(os.environ.get('DNT_EPHEMERAL_TOKEN', '') + '\\n')\n"
        )
        askpass_script.write_text(askpass_content, encoding="utf-8")
        askpass_script.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)  # 0700

        clone_env = os.environ.copy()
        clone_env["GIT_ASKPASS"] = str(askpass_script)
        clone_env["GIT_TERMINAL_PROMPT"] = "0"
        clone_env["DNT_EPHEMERAL_TOKEN"] = token

        try:
            auth_res = subprocess.run(
                ["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
                env=clone_env,
                capture_output=True,
                text=True,
            )
            if auth_res.returncode != 0:
                clean_err = auth_res.stderr.replace(token, "[REDACTED]")
                raise RuntimeError(f"Git clone failed (exit code {auth_res.returncode}): {clean_err}")
        finally:
            # Always unlink helper and purge token from in-memory environment
            askpass_script.unlink(missing_ok=True)
            if "DNT_EPHEMERAL_TOKEN" in clone_env:
                del clone_env["DNT_EPHEMERAL_TOKEN"]
            del token

        print("[OK] Repository successfully cloned via secure non-persistent authentication.")
    elif res.returncode == 0:
        print("[OK] Repository successfully cloned via public access.")
    else:
        raise RuntimeError(f"Clone failed: {res.stderr}")
else:
    print(f"Repository already present on local disk: {REPO_DIR}")

# Install compatible dependencies into active Colab Python kernel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6.0", "pandas>=2.0", "pytest>=7.0"], check=True)

# Add repository to sys.path and switch working directory
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(str(REPO_DIR))
git_head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"[OK] Active repository: {REPO_DIR} (Commit: {git_head})")


Cloning DNT-Organoid-AI repository (feature/fastq-sra-real-data-validation) to local SSD...
[OK] Repository successfully cloned via public access.
[OK] Active repository: /content/DNT-Organoid-AI (Commit: 3137616)


### Stage 4: Google Drive Account Quota & Available Headroom Verification
Now that the repository modules are loaded, verify available Google Drive storage.
Account-level capacity must be established via authorized Google Drive API metadata or explicit user confirmation of available (unused) capacity (checked at https://drive.google.com/drive/quota). Hardcoded assumptions and folder-based scans are strictly prohibited.

In [7]:
from src.quantification_bridge.checkpoints import check_storage_capacity, UnverifiedStorageQuotaError

# Verification of actual available Google Drive storage
# The user has confirmed 5 TB total capacity. We must establish actual available unused capacity.
drive_service = None
user_free_gb = None

# Method A: Attempt authorized API retrieval if user authenticated with Google Cloud
try:
    from google.colab import auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    drive_service = build('drive', 'v3')
    print("[OK] Successfully initialized authorized Google Drive v3 API client.")
except Exception as e:
    print(f"[NOTE] Google Drive API authentication skipped or unavailable: {e}")

# Method B: Prompt user for actual available space if API client not active
if drive_service is None:
    # If you prefer to specify non-interactively, set user_free_gb here (e.g. 3500.0)
    prompt_msg = (
        "Please check your available Google Drive storage at https://drive.google.com/drive/quota\n"
        "Enter confirmed available (unused) storage in GB [e.g. 3500]: "
    )
    try:
        val = input(prompt_msg).strip()
        user_free_gb = float(val) if val else None
    except Exception:
        user_free_gb = None

# Execute verified capacity check with 10 GB safety margin
cap = check_storage_capacity(
    target_path=PERSISTENCE_ROOT,
    required_bytes=35 * (1024**3),  # Estimated dense index archive (~28 GB) + margin
    safety_margin_gb=10.0,
    google_drive_service=drive_service,
    user_confirmed_account_free_gb=user_free_gb,
)

print("=" * 80)
print(f"Persistent Storage Path:       {cap['probe_path']}")
print(f"Quota Source:                  {cap['quota_source']}")
print(f"Verified Available Headroom:   {cap['free_gb']} GB")
print(f"Preflight Capacity Status:     SUFFICIENT (Ready for index persistence)")
print("=" * 80)


[OK] Successfully initialized authorized Google Drive v3 API client.


Persistent Storage Path:       /content/drive/MyDrive/DNT_Organoid_AI_Persistence
Quota Source:                  AUTHORIZED_GOOGLE_DRIVE_API
Verified Available Headroom:   221.57 GB
Preflight Capacity Status:     SUFFICIENT (Ready for index persistence)


In [9]:
import os
import sys
import subprocess
import importlib

# Install the missing SRA Toolkit in the existing environment.
subprocess.run([
    "/content/micromamba/bin/micromamba",
    "install", "-y",
    "-p", "/content/dnt_env",
    "-c", "conda-forge",
    "-c", "bioconda",
    "sra-tools=3.1.1"
], check=True)

# Make installed bioinformatics tools available to Python.
os.environ["PATH"] = (
    "/content/dnt_env/bin:" + os.environ["PATH"]
)

# Update the existing cloned repository to the fixes.
subprocess.run([
    "git", "-C", "/content/DNT-Organoid-AI",
    "pull", "--ff-only", "origin",
    "feature/fastq-sra-real-data-validation"
], check=True)

# Refresh the module containing the preflight check.
import src.quantification_bridge.profiles as profiles
importlib.reload(profiles)

# Verify every required tool individually.
for tool in [
    "STAR", "fastp", "featureCounts",
    "samtools", "infer_experiment.py", "fasterq-dump"
]:
    import shutil
    location = shutil.which(tool)
    print(f"{tool}: {location}")
    assert location, f"{tool} is missing"

STAR: /content/dnt_env/bin/STAR
fastp: /content/dnt_env/bin/fastp
featureCounts: /content/dnt_env/bin/featureCounts
samtools: /content/dnt_env/bin/samtools
infer_experiment.py: /content/dnt_env/bin/infer_experiment.py
fasterq-dump: /content/dnt_env/bin/fasterq-dump


### Stage 5: Hardware Baseline & Execution Profile Preflight
Verify that the Colab runtime meets the `colab_highmem` execution profile requirements (>= 75 GB RAM, >= 8 cores, >= 150 GB scratch disk).
Execute the full quantification bridge test suite (`tests/quantification_bridge/`) to confirm 100% test passing before beginning computation.

In [10]:
from src.quantification_bridge.pipeline import QuantificationBridgePipeline
from src.quantification_bridge.profiles import resolve_pipeline_paths

pipeline = QuantificationBridgePipeline(repo_root=REPO_DIR, profile_name="colab_highmem")
env_metrics = pipeline.record_environment_metrics()

print("=" * 80)
print("HARDWARE BASELINE AUDIT:")
print(f"OS / Kernel:           {env_metrics['uname']}")
print(f"Physical RAM:          {env_metrics['physical_ram_gb']} GB (Requirement: >= 75.0 GB)")
print(f"CPU Cores:             {env_metrics['cpu_cores']} (Requirement: >= 8)")
print(f"Local Scratch Free:    {env_metrics['scratch_disk_free_gb']} GB (Requirement: >= 150.0 GB)")
print("=" * 80)

preflight_res = pipeline.run_profile_checks(enforce_physical_ram=True)
print(f"Preflight Compatibility: {preflight_res.is_compatible}")
print(f"Fatal Errors:            {preflight_res.has_fatal_errors}")
for finding in preflight_res.findings:
    print(f"  [{finding.severity}] {finding.code}: {finding.message}")

if preflight_res.has_fatal_errors:
    raise RuntimeError("Hardware / environment preflight check failed with fatal errors! Halting execution.")

# Run full quantification bridge automated test suite
print("\nRunning quantification bridge preflight test suite...")
test_res = subprocess.run([sys.executable, "-m", "pytest", "tests/quantification_bridge/", "-v"], capture_output=False)
if test_res.returncode != 0:
    raise RuntimeError(f"Quantification bridge test suite failed (exit code {test_res.returncode})! Halting execution.")
print("[OK] All preflight tests and hardware constraints verified successfully.")


HARDWARE BASELINE AUDIT:
OS / Kernel:           Linux d38bd6ed5591 6.6.159+ #1 SMP Thu Oct 22 03:07:54 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
Physical RAM:          83.47 GB (Requirement: >= 75.0 GB)
CPU Cores:             12 (Requirement: >= 8)
Local Scratch Free:    232.72 GB (Requirement: >= 150.0 GB)
Preflight Compatibility: True
Fatal Errors:            False
  [WARNING] low_disk_space_full_study: Free storage (232.7 GB) satisfies the 2-sample pilot (70.0 GB), but is below the full 12-sample study requirement (270.0 GB).

Running quantification bridge preflight test suite...
[OK] All preflight tests and hardware constraints verified successfully.


### Stage 6: Reference Acquisition & Cryptographic Verification
Acquire the official Ensembl release 112 references directly from Ensembl HTTP/FTP servers:
- **Primary Assembly FASTA:** `Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz` (SHA-256: `d8c3af00...`)
- **Annotation GTF:** `Homo_sapiens.GRCh38.112.gtf.gz` (SHA-256: `8c87436b...`)
- **Derived BED12:** `Homo_sapiens.GRCh38.112.bed` (SHA-256: `4d1231bb...`)

Downloads are performed atomically using chunked streaming into temporary `.tmp` files. Cheksums are verified before file replacement; failed or partial downloads are deleted immediately.

In [11]:
from src.quantification_bridge.checkpoints import CheckpointManager

paths = resolve_pipeline_paths(REPO_DIR)
checkpoint_mgr = CheckpointManager(
    local_checkpoint_dir=paths["data_dir"] / "checkpoints",
    persistent_checkpoint_dir=PERSISTENCE_ROOT / "checkpoints",
)

print("Acquiring and cryptographically verifying Ensembl 112 GRCh38 references...")
ref_res = pipeline.acquire_and_verify_references()

print("[OK] Reference integrity verified against official reference manifest:")
print(f"  FASTA SHA-256: {ref_res['fasta_sha256']} (VERIFIED MATCH)")
print(f"  GTF SHA-256:   {ref_res['gtf_sha256']} (VERIFIED MATCH)")
print(f"  BED12 SHA-256: {ref_res['bed12_sha256']} (VERIFIED MATCH)")

checkpoint_mgr.record_stage_completed("REFERENCES_VERIFIED", artifacts=[ref_res])


Acquiring and cryptographically verifying Ensembl 112 GRCh38 references...
Acquiring FASTA from https://ftp.ensembl.org/pub/release-112/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz...
[Chunk 3] downloaded successfully (110245511 bytes).
[Chunk 2] downloaded successfully (110245511 bytes).
[Chunk 6] downloaded successfully (110245511 bytes).
[Chunk 5] downloaded successfully (110245511 bytes).
[Chunk 1] downloaded successfully (110245511 bytes).
[Chunk 0] downloaded successfully (110245511 bytes).
[Chunk 7] downloaded successfully (110245504 bytes).
[Chunk 4] downloaded successfully (110245511 bytes).
All chunks downloaded in 158.75s (5.30 MB/s). Merging...
Verifying SHA-256 checksum...
[OK] Merged and verified: /content/DNT-Organoid-AI/references/ensembl_112/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz (SHA-256: d8c3af0094a7bba6125763bad779ec18a81483c739c6ed122094bdf86c187b92)
POSIX sum audit: sum=22450, blocks=861294 (Expected: 22450 861294)
[OK] Reference F

### Stage 7: Canonical Dense STAR Reference-Index Generation on Local High-Speed SSD
Build the dense STAR reference index (`threads=12`, `sparseD=1`, `SAindexNbases=14`, `limitRAM=60GB`) on local NVMe/SSD.
If a verified index archive (`star_index_GRCh38_112_dense.tar`) and its companion manifest (`.manifest.json`) are already present in Google Drive, the index is restored and verified into local SSD in 1-2 minutes, bypassing 30+ minutes of re-indexing. Archives without companion manifests are never treated as verified.

In [ ]:
import time
from src.quantification_bridge.checkpoints import restore_and_verify_archive, ManifestVerificationError

PERSISTENT_REF_DIR = PERSISTENCE_ROOT / "references"
PERSISTENT_ARCHIVE = PERSISTENT_REF_DIR / "star_index_GRCh38_112_dense.tar"
PERSISTENT_MANIFEST = PERSISTENT_REF_DIR / "star_index_GRCh38_112_dense.tar.manifest.json"
SKIP_IF_ALREADY_ON_DRIVE = True

need_to_build = True
if SKIP_IF_ALREADY_ON_DRIVE and PERSISTENT_ARCHIVE.is_file() and PERSISTENT_MANIFEST.is_file():
    print(f"[FOUND] Verified index archive and companion manifest detected on Google Drive: {PERSISTENT_ARCHIVE.name}")
    print("Restoring and cryptographically verifying all constituent files into local SSD (/content)... (approx 1-2 minutes)")
    try:
        rest_res = restore_and_verify_archive(
            archive_path=PERSISTENT_ARCHIVE,
            target_extract_dir=paths["star_index_dir"],
            require_manifest=True,
        )
        print(f"[RESTORED] Extracted and verified {rest_res['verified_files_count']} files in {rest_res['extract_seconds']}s.")
        need_to_build = False
    except Exception as e:
        print(f"[INTEGRITY WARNING] Restoration failed: {e}. Building fresh dense index.")
elif SKIP_IF_ALREADY_ON_DRIVE and PERSISTENT_ARCHIVE.is_file() and not PERSISTENT_MANIFEST.is_file():
    print(f"[WARNING] Archive {PERSISTENT_ARCHIVE.name} exists, but companion manifest is missing! Cannot treat as verified. Will regenerate index.")

if need_to_build:
    print("Starting canonical dense STAR genomeGenerate on local SSD (threads=12, limit_ram=60GB)...")
    t0 = time.time()
    idx_res = pipeline.build_star_index(
        threads=12,
        sparse_d=1,  # Canonical dense reference standard on Colab High-RAM
        genome_sa_index_nbases=14,
        limit_ram=60000000000,
    )
    duration_min = (time.time() - t0) / 60
    print(f"[OK] STAR Index generation finished cleanly in {duration_min:.2f} minutes!")


Starting canonical dense STAR genomeGenerate on local SSD (threads=12, limit_ram=60GB)...
Building STAR genome index at /content/DNT-Organoid-AI/references/ensembl_112/star_index (threads=12, sparse_d=1, nbases=14, limit_ram=60000000000)...
Decompressing /content/DNT-Organoid-AI/references/ensembl_112/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz to /content/DNT-Organoid-AI/references/ensembl_112/Homo_sapiens.GRCh38.dna.primary_assembly.fa for STAR...
Decompressing /content/DNT-Organoid-AI/data/raw/references/Homo_sapiens.GRCh38.112.gtf.gz to /content/DNT-Organoid-AI/data/raw/references/Homo_sapiens.GRCh38.112.gtf for STAR...
Executing STAR genomeGenerate:
STAR --runMode genomeGenerate --genomeDir /content/DNT-Organoid-AI/references/ensembl_112/star_index --genomeFastaFiles /content/DNT-Organoid-AI/references/ensembl_112/Homo_sapiens.GRCh38.dna.primary_assembly.fa --sjdbGTFfile /content/DNT-Organoid-AI/data/raw/references/Homo_sapiens.GRCh38.112.gtf --sjdbOverhang 149 --genomeSAsparse

### Stage 8: Index Artifact Inventory & Process Load Validation
Perform dual-layer index verification:
1. **Structural Validation:** Inventory all 14 binary index files, non-zero sizes, chromosome count, and `genomeParameters.txt` parsing.
2. **Process Load Validation:** Dry-run process address-space loading via `--genomeLoad NoSharedMemory` and `--outSAMtype None` on a synthetic probe. This verifies index mapping without SysV shared-memory `/dev/shm` limitations.

In [ ]:
from scripts.run_colab_reference_indexing import verify_index_artifacts, lightweight_index_load_test

print("Verifying index artifact inventory and structural parameters...")
eval_res = verify_index_artifacts(paths["star_index_dir"])
if not eval_res["is_complete"]:
    raise RuntimeError(f"Index structural validation failed! Missing: {eval_res['missing_files']}")

print(f"[OK] All {len(eval_res['file_details'])} required files verified present and non-zero:")
for fname, detail in eval_res['file_details'].items():
    print(f"  - {fname:<25} {detail['size_mb']:>9.2f} MB")

print(f"\nTotal Local Index Size: {eval_res['total_index_gb']} GB")
print(f"Chromosomes Indexed:    {eval_res['chromosome_count']}")
print(f"Genome Parameters:      {eval_res['genome_parameters']}")

# Process-level dry-run load test (NoSharedMemory mode, immune to /dev/shm container limits)
print("\nExecuting process-level index load test (NoSharedMemory)...")
load_test = lightweight_index_load_test(paths["star_index_dir"], star_bin="STAR")
print(f"Index Load Validation Status: {load_test['load_test_passed']} (Mode: {load_test.get('index_load_mode')}, Time: {load_test.get('load_time_seconds')}s)")
if not load_test["load_test_passed"]:
    raise RuntimeError(f"Process-level index load test failed! Error: {load_test.get('error')}")

checkpoint_mgr.record_stage_completed(
    "STAR_INDEX_BUILT_AND_VERIFIED",
    artifacts=[{"dir": str(paths["star_index_dir"]), "total_gb": eval_res["total_index_gb"]}],
    metrics={"load_test": load_test},
)


### Stage 9: Single-Archive Persistent Storage Synchronization
Package the completed STAR index into a single verified tar archive (`star_index_GRCh38_112_dense.tar`) with a companion cryptographic manifest recording every constituent file's SHA-256. Synchronize to Google Drive with verified capacity checking.

In [ ]:
from src.quantification_bridge.checkpoints import archive_and_sync_directory

TARGET_PERSISTENT_REF = PERSISTENCE_ROOT / "references"
SAFETY_MARGIN_GB = 10.0

print("Archiving and synchronizing index to Google Drive with companion cryptographic manifest...")
archive_res = archive_and_sync_directory(
    src_dir=paths["star_index_dir"],
    dst_dir=TARGET_PERSISTENT_REF,
    archive_name="star_index_GRCh38_112_dense.tar",
    safety_margin_gb=SAFETY_MARGIN_GB,
    google_drive_service=drive_service,
    user_confirmed_account_free_gb=user_free_gb,
    delete_source_after=False,  # Retain local copy on fast SSD for active alignment
)

print("=" * 80)
print(f"[PERSISTENCE VERIFIED] Successfully archived {archive_res['total_files']} files to Google Drive in {archive_res['total_seconds']}s.")
print(f"Destination Archive:  {archive_res['dst_archive_path']} ({archive_res['archive_size_gb']} GB)")
print(f"Companion Manifest:   {archive_res['dst_manifest_path']}")
print(f"Archive SHA-256:      {archive_res['archive_sha256']}")
print("=" * 80)

checkpoint_mgr.record_stage_completed(
    "STAR_INDEX_PERSISTED_TO_GDRIVE",
    artifacts=[{
        "archive": archive_res["dst_archive_path"],
        "manifest": archive_res["dst_manifest_path"],
        "sha256": archive_res["archive_sha256"],
    }],
)


## MANDATORY SCIENTIFIC REVIEW GATE
### EXECUTION HALT

In strict compliance with user governance:
- **Reference Indexing is Complete and Validated.**
- **Persistent Backup is Cryptographically Verified on Google Drive with Companion Manifest.**
- **DO NOT proceed to FASTQ sequencing library acquisition or alignment until this report is formally reviewed and approved.**
- Run the final cell below to produce the formal reproducibility audit table.

In [ ]:
import subprocess
import shutil

git_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
star_ver = subprocess.check_output(["STAR", "--version"], text=True).strip() if shutil.which("STAR") else "STAR 2.7.11b"

print("=" * 80)
print("DNT ORGANOID AI - REFERENCE INDEX REPRODUCIBILITY AUDIT")
print("=" * 80)
print(f"Repository Git Commit:        {git_hash}")
print(f"Active Profile:               colab_highmem")
print(f"Reference Genome:             Ensembl GRCh38 release 112 (Homo sapiens)")
print(f"STAR Version:                 {star_ver}")
print(f"Index Parameters:             sparseD=1 (dense), SAindexNbases=14, limitRAM=60GB")
print(f"Total Local Index Size:       {eval_res['total_index_gb']} GB")
print(f"Google Drive Archive:         {archive_res['archive_size_gb']} GB (Verified Match)")
print(f"Archive SHA-256:              {archive_res['archive_sha256']}")
print(f"Constituent Files Verified:   {len(eval_res['file_details'])} (All Present & Non-Zero)")
print(f"Index Structural Validation:  PASSED")
print(f"Process Load Validation:      PASSED (Mode: NoSharedMemory, Time: {load_test.get('load_time_seconds')}s)")
print(f"Google Drive Checkpoint:      COMPLETED & PERSISTED WITH MANIFEST")
print(f"Status:                       AWAITING_OWNER_PILOT_FASTQ_AUTHORIZATION")
print("=" * 80)
print("MANDATORY SCIENTIFIC REVIEW GATE: Execution strictly HALTED as governed.")
print("DO NOT download FASTQ sequencing libraries or launch alignment until this report is approved.")
